# Ensemble Federated Learning with Clustering on Rotation-Based CIFAR-10

This notebook implements the ensemble federated learning approach with client clustering based on weight differences.

**Method Overview:**
1. Warmup phase: Clients train locally and collect weight differences
2. Clustering: Group clients based on weight difference patterns
3. Ensemble training: K specialized feature extractors (one per cluster) + shared classifier
4. Hierarchical aggregation: Within-cluster averaging for features, global averaging for classifier

In [ ]:
# Setup for Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
    
    from google.colab import drive
    drive.mount('/content/drive')
    
    import os
    os.chdir('/content/drive/MyDrive/EnsembleFederatedLearning')
    
    !pip install -q torch torchvision scikit-learn matplotlib seaborn scipy
    
except ImportError:
    IN_COLAB = False
    print("Running locally")

## Import Libraries

In [ ]:
import sys
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, confusion_matrix
from scipy.optimize import linear_sum_assignment
import copy
import random
import time

sys.path.append('..')
from training.ensemble_fl import EnsembleFedAvg
from training.utils import get_model, set_seed

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load Configuration

## ⚠️ Memory-Saving Tips for Google Colab

**If the notebook crashes due to memory issues, try these fixes in `config.json`:**

```json
{
    "batch_size": 32,           // Reduce from 64 → saves ~50% memory per batch
    "num_clients": 25,          // Reduce from 50 → less weight storage
    "num_clusters_model": 4,    // Keep at 4 (or reduce to 2)
    "training_rounds": 20,      // Reduce from 30
    "warmup_epochs": 1          // Reduce from 2 → faster, less memory
}
```

**Why crashes happen:**
- **4 ResNet18 models** loaded simultaneously (~44MB each = 176MB)
- **50 clients × weight differences** stored in RAM
- **Colab Free**: 12-15GB RAM limit

**Solutions:**
1. Reduce batch size and number of clients (above)
2. Use **Colab Pro** (more RAM)
3. Run locally if possible

In [ ]:
# Load configuration from JSON
with open('experiments/config.json', 'r') as f:
    CONFIG = json.load(f)

# Set random seeds
SEED = CONFIG.get('seed', 42)
set_seed(SEED)
CONFIG['seed'] = SEED

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## Create Rotation-Based Dataset

In [ ]:
class RotatedCIFAR10Dataset(Dataset):
    """CIFAR-10 dataset with rotation applied."""
    def __init__(self, base_dataset, rotation_angle):
        self.base_dataset = base_dataset
        self.rotation_angle = rotation_angle
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]
        
        if self.rotation_angle != 0:
            image = transforms.functional.rotate(image, self.rotation_angle)
        
        image = self.transform(image)
        return image, label

# Load CIFAR-10
print("Loading CIFAR-10 dataset...")
train_dataset_raw = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=None)
test_dataset_raw = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=None)

print(f"Train dataset size: {len(train_dataset_raw)}")
print(f"Test dataset size: {len(test_dataset_raw)}")

## Distribute Data Across Clients with Rotations

In [ ]:
# Generate rotation angles dynamically
rotation_angles = [int(360 * i / CONFIG['num_rotation_clusters']) for i in range(CONFIG['num_rotation_clusters'])]
print(f"Rotation angles: {rotation_angles}°")

# Assign clients to rotations
clients_per_rotation = CONFIG['num_clients'] // CONFIG['num_rotation_clusters']
client_rotation_labels = []

for rotation_idx, angle in enumerate(rotation_angles):
    start_client = rotation_idx * clients_per_rotation
    end_client = start_client + clients_per_rotation
    
    if rotation_idx == len(rotation_angles) - 1:
        end_client = CONFIG['num_clients']
    
    for client_idx in range(start_client, end_client):
        client_rotation_labels.append(angle)

print(f"\nClient distribution across rotations:")
for angle in rotation_angles:
    count = client_rotation_labels.count(angle)
    print(f"  {angle}°: {count} clients")

# Create rotated datasets for each client
train_subsets = []
samples_per_client = len(train_dataset_raw) // CONFIG['num_clients']
all_indices = list(range(len(train_dataset_raw)))
random.shuffle(all_indices)

for client_idx in range(CONFIG['num_clients']):
    angle = client_rotation_labels[client_idx]
    rotated_dataset = RotatedCIFAR10Dataset(train_dataset_raw, angle)
    
    start_idx = client_idx * samples_per_client
    end_idx = start_idx + samples_per_client if client_idx < CONFIG['num_clients'] - 1 else len(train_dataset_raw)
    
    client_indices = all_indices[start_idx:end_idx]
    subset = Subset(rotated_dataset, client_indices)
    train_subsets.append(subset)

print(f"\nCreated {len(train_subsets)} client datasets")
print(f"Average samples per client: {np.mean([len(s) for s in train_subsets]):.1f}")

## Create Test Dataset

In [ ]:
# Create test dataset (no rotation)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print(f"Test dataset size: {len(test_dataset)}")

## Initialize Ensemble FL and Run Warmup

Initialize the ensemble FL system and run warmup phase to collect weight differences from clients.

In [ ]:
print("="*70)
print("INITIALIZING ENSEMBLE FEDERATED LEARNING")
print("="*70)

# Memory optimization: Clear any cached data
import gc
torch.cuda.empty_cache() if torch.cuda.is_available() else None
gc.collect()

# Check available memory
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"GPU Memory Available: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9:.2f} GB")

# Initialize EnsembleFedAvg
ensemble_fl = EnsembleFedAvg(
    train_subsets=train_subsets,
    test_set=test_dataset,
    num_clients=CONFIG['num_clients'],
    device=device,
    model_name=CONFIG['model_name'],
    pretrained=CONFIG.get('pretrained', False),
    num_clusters=CONFIG['num_clusters_model'],
    batch_size=CONFIG['batch_size'],
    lr=CONFIG['lr'],
    seed=CONFIG['seed']
)

print(f"\nEnsemble FL initialized:")
print(f"  Clients: {CONFIG['num_clients']}")
print(f"  Clusters: {CONFIG['num_clusters_model']}")
print(f"  Model: {CONFIG['model_name']}")

# Run warmup phase
print(f"\n{'='*70}")
print("WARMUP PHASE: Collecting Weight Differences")
print("="*70)

warmup_start = time.time()
ensemble_fl.run_warmup(
    use_fedavg=False, 
    local_epochs=CONFIG['warmup_epochs'],
    use_weight_diff=True
)
warmup_time = time.time() - warmup_start

print(f"\nWarmup completed in {warmup_time:.2f}s ({warmup_time/60:.2f} min)")
print(f"Weight differences collected from {CONFIG['num_clients']} clients")

# Clear memory after warmup
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## Perform Client Clustering

Cluster clients based on their weight difference patterns using K-Means.

In [ ]:
print(f"\n{'='*70}")
print("CLUSTERING PHASE: Grouping Clients")
print("="*70)

# Extract weight differences (FC + Layer4)
print("Extracting weight differences from FC + Layer4 layers...")
layer_grads = ensemble_fl.get_client_layer_gradients(average_across_epochs=True)

gradients = []
for client_idx in range(ensemble_fl.num_clients):
    client_grads = layer_grads[client_idx]
    selected_grads = []
    for name, grad in client_grads.items():
        if 'fc' in name or 'layer4' in name:
            selected_grads.append(grad)
    if selected_grads:
        gradients.append(np.concatenate(selected_grads))

gradient_matrix = np.array(gradients)
print(f"Weight difference matrix shape: {gradient_matrix.shape}")

# Perform K-Means clustering
print(f"\nApplying K-Means clustering (k={CONFIG['num_clusters_model']})...")
kmeans = KMeans(
    n_clusters=CONFIG['num_clusters_model'], 
    random_state=CONFIG['seed'], 
    n_init=10
)
predicted_clusters = kmeans.fit_predict(gradient_matrix)

# Calculate clustering quality metrics
silhouette = silhouette_score(gradient_matrix, predicted_clusters)

print(f"\nClustering Results:")
print(f"  Silhouette Score: {silhouette:.4f}")
print(f"\nCluster distribution:")
for k in range(CONFIG['num_clusters_model']):
    count = np.sum(predicted_clusters == k)
    print(f"  Cluster {k}: {count} clients")

# Assign clusters to ensemble
ensemble_fl.client_clusters = {i: int(predicted_clusters[i]) for i in range(CONFIG['num_clients'])}

## Evaluate Clustering Quality

Compare predicted clusters with ground truth rotation assignments.

In [ ]:
# Ground truth mapping
rotation_to_id = {angle: idx for idx, angle in enumerate(rotation_angles)}
true_clusters = np.array([rotation_to_id[angle] for angle in client_rotation_labels])

# Calculate Adjusted Rand Index
ari = adjusted_rand_score(true_clusters, predicted_clusters)

# Create confusion matrix
confusion = np.zeros((CONFIG['num_clusters_model'], CONFIG['num_clusters_model']), dtype=int)
for true_label, pred_label in zip(true_clusters, predicted_clusters):
    confusion[true_label, pred_label] += 1

# Calculate alignment accuracy using Hungarian algorithm
cost_matrix = -confusion
row_ind, col_ind = linear_sum_assignment(cost_matrix)
alignment_accuracy = confusion[row_ind, col_ind].sum() / len(true_clusters)

print(f"\nClustering Quality vs Ground Truth:")
print(f"  Adjusted Rand Index: {ari:.4f}")
print(f"  Alignment Accuracy: {alignment_accuracy:.2%}")

# Visualize confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(confusion, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'Pred {i}' for i in range(CONFIG['num_clusters_model'])],
            yticklabels=[f'True {i} ({rotation_angles[i]}°)' for i in range(CONFIG['num_clusters_model'])])
plt.title(f'Clustering Alignment with True Rotations\nARI: {ari:.4f}, Accuracy: {alignment_accuracy:.2%}')
plt.ylabel('True Rotation Cluster')
plt.xlabel('Predicted Cluster')
plt.tight_layout()
plt.savefig('ensemble_clustering_confusion.png', dpi=300, bbox_inches='tight')
plt.show()

# Cluster purity analysis
print("\nCluster Purity Analysis:")
for pred_cluster in range(CONFIG['num_clusters_model']):
    mask = predicted_clusters == pred_cluster
    if np.sum(mask) > 0:
        true_labels_in_cluster = true_clusters[mask]
        unique, counts = np.unique(true_labels_in_cluster, return_counts=True)
        dominant_label = unique[np.argmax(counts)]
        purity = np.max(counts) / np.sum(mask)
        print(f"  Cluster {pred_cluster}: Dominant={rotation_angles[dominant_label]}°, Purity={purity:.2%}")

## Initialize and Train Ensemble

Initialize the ensemble with K feature extractors and shared classifier, then train.

In [ ]:
print(f"\n{'='*70}")
print("ENSEMBLE TRAINING PHASE")
print("="*70)

# Initialize ensemble architecture
print("\nInitializing ensemble architecture...")
ensemble_fl._initialize_ensemble()
print(f"  Feature extractors: {CONFIG['num_clusters_model']} (ResNet backbones)")
print(f"  Shared classifier: 1 ({CONFIG['num_clusters_model']} × 512 → 10 classes)")

# Training configuration
num_rounds = CONFIG.get('training_rounds', 30)
client_fraction = CONFIG.get('client_fraction', 0.2)

print(f"\nTraining for {num_rounds} rounds...")
print(f"  Client fraction: {client_fraction} ({int(client_fraction * CONFIG['num_clients'])} clients/round)")
print(f"  Local epochs: 1")

# Storage for results
test_losses = []
test_accs = []
round_times = []

training_start = time.time()

for round_num in range(1, num_rounds + 1):
    round_start = time.time()
    
    # Train one round
    test_loss, test_acc = ensemble_fl.train_ensemble_round(
        round_num=round_num,
        fraction=client_fraction,
        local_epochs=1
    )
    
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    round_time = time.time() - round_start
    round_times.append(round_time)
    
    # Print progress
    if round_num % 5 == 0 or round_num == 1:
        print(f"Round {round_num}/{num_rounds} - "
              f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}, "
              f"Time: {round_time:.2f}s")

total_training_time = time.time() - training_start

print(f"\n{'='*70}")
print("ENSEMBLE TRAINING COMPLETE")
print("="*70)
print(f"Total time (warmup + clustering + training): {warmup_time + total_training_time:.2f}s")
print(f"Training time only: {total_training_time:.2f}s ({total_training_time/60:.2f} min)")
print(f"Average time per round: {np.mean(round_times):.2f}s")
print(f"Final test accuracy: {test_accs[-1]:.4f}")
print(f"Best test accuracy: {max(test_accs):.4f} (round {np.argmax(test_accs)+1})")

## Visualize Training Progress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Loss curve
ax = axes[0]
rounds_range = range(1, num_rounds + 1)
ax.plot(rounds_range, test_losses, 'o-', linewidth=2, markersize=4, color='purple')
ax.set_xlabel('Round', fontsize=12)
ax.set_ylabel('Test Loss', fontsize=12)
ax.set_title('Ensemble Test Loss Over Rounds', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 2: Accuracy curve
ax = axes[1]
ax.plot(rounds_range, test_accs, 's-', linewidth=2, markersize=4, color='green')
ax.axhline(y=max(test_accs), color='red', linestyle='--', alpha=0.5, 
           label=f'Best: {max(test_accs):.4f}')
ax.set_xlabel('Round', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Ensemble Test Accuracy Over Rounds', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('ensemble_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("Training curves saved as 'ensemble_training_curves.png'")

## Test on Different Rotations

In [ ]:
# Create test loaders for each rotation
rotation_test_results = {}

print(f"Evaluating ensemble on different rotations...\n")

for angle in rotation_angles:
    rotated_test = RotatedCIFAR10Dataset(test_dataset_raw, angle)
    test_subset_rot = Subset(rotated_test, list(range(len(test_dataset_raw))))
    rotation_loader = DataLoader(test_subset_rot, batch_size=CONFIG['batch_size'], shuffle=False)
    
    # Evaluate
    test_loss, test_acc = ensemble_fl.evaluate_ensemble()
    rotation_test_results[angle] = test_acc
    print(f"Rotation {angle}°: Accuracy = {test_acc:.4f}")

# Visualize per-rotation performance
plt.figure(figsize=(10, 6))
angles = list(rotation_test_results.keys())
accs = list(rotation_test_results.values())

plt.bar([f"{a}°" for a in angles], accs, alpha=0.7, edgecolor='black', color='mediumseagreen')
plt.axhline(y=np.mean(accs), color='red', linestyle='--', label=f'Average: {np.mean(accs):.4f}')
plt.xlabel('Rotation Angle', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('Ensemble Performance on Different Rotations', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.ylim([0, 1])
plt.tight_layout()
plt.savefig('ensemble_rotation_performance.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPerformance variance across rotations:")
print(f"  Mean: {np.mean(accs):.4f}")
print(f"  Std:  {np.std(accs):.4f}")
print(f"  Min:  {np.min(accs):.4f} ({angles[np.argmin(accs)]}°)")
print(f"  Max:  {np.max(accs):.4f} ({angles[np.argmax(accs)]}°)")

## Confusion Matrix on Test Set

In [ ]:
# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

# Collect all predictions and true labels
all_predictions = []
all_labels = []

ensemble_model = ensemble_fl.get_ensemble_model()
ensemble_model.eval()

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = ensemble_model(inputs)
        _, predicted = outputs.max(1)
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Compute confusion matrix
cm = confusion_matrix(all_labels, all_predictions)

# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: Confusion matrix with counts
ax = axes[0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.setp(ax.get_yticklabels(), rotation=0)

# Plot 2: Normalized confusion matrix (percentages)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
ax = axes[1]
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Greens', ax=ax,
            xticklabels=class_names, yticklabels=class_names,
            vmin=0, vmax=1, cbar_kws={'label': 'Proportion'})
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.setp(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('ensemble_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Analyze per-class performance
print("\nPer-class Performance:")
print(f"{'Class':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
print("="*65)

for i, class_name in enumerate(class_names):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    support = cm[i, :].sum()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"{class_name:<15} {precision:<12.4f} {recall:<12.4f} {f1:<12.4f} {support:<10}")

overall_accuracy = np.trace(cm) / cm.sum()
print(f"\n{'Overall Accuracy':<15} {overall_accuracy:.4f}")

# Find most confused pairs
print("\nMost Confused Class Pairs:")
confused_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i, j] > 0:
            confused_pairs.append((class_names[i], class_names[j], cm[i, j]))

confused_pairs.sort(key=lambda x: x[2], reverse=True)
for true_class, pred_class, count in confused_pairs[:10]:
    print(f"  {true_class:<12} → {pred_class:<12}: {count:>4} times")

## Save Results

In [ ]:
# Save results to file
results = {
    'method': 'ensemble_clustering',
    'config': CONFIG,
    'warmup_time': warmup_time,
    'training_time': total_training_time,
    'total_time': warmup_time + total_training_time,
    'num_rounds': num_rounds,
    'clustering': {
        'silhouette_score': float(silhouette),
        'adjusted_rand_index': float(ari),
        'alignment_accuracy': float(alignment_accuracy),
        'cluster_distribution': {int(k): int(np.sum(predicted_clusters == k)) 
                                for k in range(CONFIG['num_clusters_model'])}
    },
    'final_test_acc': float(test_accs[-1]),
    'best_test_acc': float(max(test_accs)),
    'test_losses': [float(x) for x in test_losses],
    'test_accs': [float(x) for x in test_accs],
    'rotation_results': {int(k): float(v) for k, v in rotation_test_results.items()}
}

with open('ensemble_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to 'ensemble_results.json'")

# Save model checkpoint
torch.save({
    'round': num_rounds,
    'cluster_models': [model.state_dict() for model in ensemble_fl.cluster_models],
    'classifier': ensemble_fl.ensemble_classifier.state_dict(),
    'client_clusters': ensemble_fl.client_clusters,
    'test_acc': test_accs[-1],
}, 'ensemble_model_checkpoint.pth')

print("Model checkpoint saved to 'ensemble_model_checkpoint.pth'")

## Summary

**Ensemble Federated Learning Results:**

**Architecture:**
- K specialized feature extractors (ResNet backbones without FC layer)
- 1 shared classifier (K × 512 → 10 classes)
- Features from active cluster models are concatenated and normalized

**Training Strategy:**
1. **Warmup**: Clients train locally, collect weight differences (Δw)
2. **Clustering**: Group clients using K-Means on weight differences
3. **Ensemble Training**: 
   - Within-cluster: Aggregate feature extractors for each cluster
   - Global: Aggregate shared classifier across all clients
4. **Inference**: Concatenate features from client's cluster models → shared classifier

**Key Advantages:**
- Specialized models adapt to data heterogeneity
- Shared classifier maintains unified decision boundary
- Weight differences naturally capture client characteristics
- Better privacy alignment (FL algorithms compute Δw)

**Comparison with Baselines:**
- vs Centralized: Privacy-preserving, handles distributed data
- vs FedAvg: Better performance on non-IID data with feature heterogeneity
- Clustering quality validated by ARI and alignment metrics

**Files Generated:**
- `ensemble_results.json` - Complete training metrics
- `ensemble_model_checkpoint.pth` - Trained ensemble
- `ensemble_training_curves.png` - Learning curves
- `ensemble_rotation_performance.png` - Per-rotation accuracy
- `ensemble_clustering_confusion.png` - Clustering quality
- `ensemble_confusion_matrix.png` - Classification performance